# Lab 7d: Measuring $M_L$, $m_b$, and $M_s$ from seismograms

> **Colab note:** This notebook is designed to run on **Google Colab**. The first code cell installs dependencies. [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/UW-geophysics-edu/ess-412-512-intro2seismology/blob/main/notebooks/07d_Magnitude_Measurement_Practice.ipynb)

## Learning objectives

By the end of this lab, students should be able to:

1. **Simulate** a Wood-Anderson seismograph response and measure $M_L$.
2. **Pick** the maximum P-wave amplitude/period and compute $m_b$ from the Gutenberg-Richter formula.
3. **Measure** a Rayleigh-wave amplitude at ~20 s period and compute $M_s$ using the Prague formula.
4. **Compare** the three magnitude estimates for the same event and explain discrepancies in terms of source spectrum and bandpass.

## Key concepts

| Magnitude | What is measured | Typical band | Formula reference |
|-----------|-----------------|-------------|-------------------|
| $M_L$ | Peak displacement on Wood-Anderson horizontal | 1–10 Hz | Richter (1935); Hutton & Boore (1987) |
| $m_b$ | Max $(A/T)$ of P on short-period vertical | 0.5–2 Hz | Gutenberg & Richter (1956) |
| $M_s$ | Max $(A/T)$ of Rayleigh wave at ~20 s | 0.04–0.065 Hz | Vaněk et al. (1962), Prague formula |

## Prerequisites

- Lecture 7c (Earthquake Magnitudes) — magnitude definitions, saturation, omega-square spectrum
- Lab 7c — toy magnitude sensitivity and catalog exploration
- Basic familiarity with ObsPy (`Stream`, `Trace`, instrument response removal)

## Notebook outline

1. [Setup & event selection](#setup)
2. [Exercise A: $M_L$ — Wood-Anderson simulation](#exercise-a)
3. [Exercise B: $m_b$ — short-period P amplitude](#exercise-b)
4. [Exercise C: $M_s$ — Rayleigh-wave amplitude](#exercise-c)
5. [Exercise D: Comparison & wrap-up](#exercise-d)

---

📖 *Shearer (2009), §9.7*: magnitude scales and their physical basis.

In [ ]:
# Install dependencies (for Google Colab or missing packages)
import subprocess, sys

def _install(pkg, pip_name=None):
    try:
        __import__(pkg)
    except ImportError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pip_name or pkg])

for pkg, pip in [('numpy','numpy'), ('matplotlib','matplotlib'),
                 ('obspy','obspy'), ('pandas','pandas')]:
    _install(pkg, pip)

print('All dependencies available \u2713')

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from obspy import UTCDateTime
from obspy.clients.fdsn import Client
from obspy.geodetics import gps2dist_azimuth
from obspy.taup import TauPyModel

import pandas as pd

# Plotting defaults
plt.rcParams['figure.figsize'] = (10, 4)
plt.rcParams['font.size'] = 11
plt.rcParams['axes.grid'] = True

# FDSN client and travel-time model
client = Client('IRIS')
model  = TauPyModel(model='iasp91')

print('Setup complete \u2713')

<a id="setup"></a>

## 1. Event and station selection

We fetch a **moderate earthquake** ($M_w$ 6–7) and a broadband station at **teleseismic distance** (30°–80°), which allows us to measure all three magnitude types on the same record.

The code below queries the IRIS FDSN catalog, selects the first suitable event, and downloads a three-component broadband waveform with instrument response metadata.

In [ ]:
# ---- Event search ----
t_end   = UTCDateTime()                      # now
t_start = t_end - 365 * 86400                # past year

cat = client.get_events(starttime=t_start, endtime=t_end,
                        minmagnitude=6.0, maxmagnitude=7.0,
                        maxdepth=70,          # shallow → good surface waves
                        orderby='magnitude',
                        limit=5)
print(f'Events found: {len(cat)}')

# Pick the first event
event  = cat[0]
origin = event.preferred_origin() or event.origins[0]
ev_mag = event.preferred_magnitude() or event.magnitudes[0]

ev_lat, ev_lon, ev_dep_km = origin.latitude, origin.longitude, origin.depth / 1e3
ev_time = origin.time

print(f'Event:  {ev_time}  lat={ev_lat:.2f}  lon={ev_lon:.2f}  depth={ev_dep_km:.1f} km')
print(f'Catalog magnitude: {ev_mag.magnitude_type} {ev_mag.mag:.1f}')

In [ ]:
# ---- Station search (teleseismic, broadband) ----
inv = client.get_stations(
    starttime=ev_time, endtime=ev_time + 3600,
    latitude=ev_lat, longitude=ev_lon,
    minradius=30, maxradius=80,         # teleseismic
    channel='BH?', level='response',
    matchtimeseries=True
)

# Pick first available station
net = inv[0].code
sta = inv[0][0].code
loc = inv[0][0][0].location_code

st_lat = float(inv[0][0].latitude)
st_lon = float(inv[0][0].longitude)

dist_m, az, baz = gps2dist_azimuth(ev_lat, ev_lon, st_lat, st_lon)
dist_km  = dist_m / 1e3
dist_deg = dist_km / 111.19

print(f'Station: {net}.{sta}  lat={st_lat:.2f}  lon={st_lon:.2f}')
print(f'Distance: {dist_km:.0f} km  ({dist_deg:.1f}°)')

In [ ]:
# ---- Download waveforms (generous window: 5 min before to 60 min after) ----
st_raw = client.get_waveforms(net, sta, loc, 'BH?',
                              ev_time - 300, ev_time + 3600)
st_raw.merge(fill_value='interpolate')
print(st_raw)

# Remove instrument response → displacement (m)
st_disp = st_raw.copy()
st_disp.detrend('linear')
st_disp.taper(0.02)
pre_filt = (0.004, 0.007, 25.0, 30.0)   # conservative pre-filter (Hz)
st_disp.remove_response(inventory=inv, output='DISP',
                        pre_filt=pre_filt, water_level=60)
print('Instrument response removed → displacement (m)')

In [ ]:
# ---- Predicted arrivals (for windowing) ----
arrivals = model.get_travel_times(source_depth_in_km=ev_dep_km,
                                  distance_in_degree=dist_deg,
                                  phase_list=['P', 'S', 'Rayleigh'])
t_P = None
for arr in arrivals:
    if arr.name == 'P':
        t_P = arr.time
        break
if t_P is None:
    # Fallback: approximate P travel time
    t_P = dist_deg * 8.0  # ~8 s/deg for teleseismic P

# Rayleigh group velocity window (~3.5–4.2 km/s)
t_R_early = dist_km / 4.2
t_R_late  = dist_km / 3.0

print(f'Predicted P arrival:      {t_P:.1f} s after origin')
print(f'Rayleigh window:          {t_R_early:.0f}–{t_R_late:.0f} s after origin')

<a id="exercise-a"></a>

---

## Exercise A: $M_L$ — Local magnitude via Wood-Anderson simulation

Richter's $M_L$ is defined as the peak trace amplitude (in mm) on a **Wood-Anderson torsion seismograph**, corrected for distance:

$$
M_L = \log_{10}(A_{\text{WA}}) - \log_{10}(A_0(\Delta))
$$

where $A_{\text{WA}}$ is the **half** peak-to-peak displacement amplitude in mm and $-\log_{10} A_0(\Delta)$ is Richter's empirical distance correction (or equivalently the Hutton & Boore (1987) revision for Southern California).

We simulate the WA instrument response digitally and measure the peak on the horizontal components.

### Predict
At teleseismic distance ($\Delta > 30°$), $M_L$ is **not designed to work**—it was calibrated for $\Delta < 600$ km.  
Do you expect our $M_L$ estimate to be reliable here? What kind of bias would you predict?

In [ ]:
# ---- Wood-Anderson simulation ----
# ObsPy's Wood-Anderson poles/zeros (standard short-period WA torsion)
PAZ_WA = {
    'poles': [-6.2832 - 4.7124j, -6.2832 + 4.7124j],
    'zeros': [0j, 0j],
    'gain': 1.0,
    'sensitivity': 2800.0   # standard WA magnification
}

# Work on horizontal components only
st_wa = st_disp.copy()
# Select horizontal channels (last char N/E or 1/2)
st_h = st_wa.select(component='[NE12]')  # may not work with regex; do manually
if len(st_h) == 0:
    st_h = st_wa.copy()
    st_h.traces = [tr for tr in st_h if tr.stats.channel[-1] in 'NE12']

# Simulate WA from displacement: first restore to velocity, then simulate WA
# Simpler approach: simulate WA from the raw data using full response chain
st_wa_sim = st_raw.copy().select(component='[NE12]')
if len(st_wa_sim) == 0:
    st_wa_sim = st_raw.copy()
    st_wa_sim.traces = [tr for tr in st_wa_sim if tr.stats.channel[-1] in 'NE12']

st_wa_sim.detrend('linear')
st_wa_sim.taper(0.02)
st_wa_sim.simulate(paz_remove=None, paz_simulate=PAZ_WA,
                   remove_sensitivity=True, simulate_sensitivity=True,
                   seedresp={'filename': None, 'units': 'DIS'},
                   pre_filt=(0.5, 1.0, 10.0, 15.0))
# Alternatively: use remove_response then convolve with WA
# A simpler and more robust approach:
st_wa_sim2 = st_disp.copy()
st_wa_sim2.traces = [tr for tr in st_wa_sim2 if tr.stats.channel[-1] in 'NE12']
st_wa_sim2.filter('bandpass', freqmin=1.0, freqmax=10.0, corners=4, zerophase=True)

# Measure peak displacement on the horizontal that gives the largest WA amplitude
peak_wa_m = 0.0
for tr in st_wa_sim2:
    half_p2p = (tr.data.max() - tr.data.min()) / 2.0
    if half_p2p > peak_wa_m:
        peak_wa_m = half_p2p

peak_wa_mm = peak_wa_m * 1e3  # convert m → mm
print(f'Peak WA half-amplitude: {peak_wa_m:.2e} m  =  {peak_wa_mm:.4f} mm')

In [ ]:
# ---- Plot WA-simulated horizontal seismogram ----
tr_plot = st_wa_sim2[0]
times = tr_plot.times(reftime=ev_time)

fig, ax = plt.subplots()
ax.plot(times, tr_plot.data * 1e3, 'k', lw=0.5)
# Mark peak amplitude
idx_max = np.argmax(np.abs(tr_plot.data))
ax.plot(times[idx_max], tr_plot.data[idx_max] * 1e3, 'rv', ms=10, label='peak')
ax.set_xlabel('Time after origin (s)')
ax.set_ylabel('Displacement (mm, WA-simulated)')
ax.set_title(f'Wood-Anderson simulated horizontal — {net}.{sta}')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# ---- Compute ML ----
# Hutton & Boore (1987) distance correction for Southern California:
#   -log10(A0) = 1.110 * log10(r/100) + 0.00189*(r - 100) + 3.0
# where r is hypocentral distance in km.
# Note: this calibration is for r < ~700 km.  At teleseismic distance
# the result is extrapolated and should be interpreted with caution.

r = dist_km
neg_logA0 = 1.110 * np.log10(r / 100.0) + 0.00189 * (r - 100.0) + 3.0

if peak_wa_mm > 0:
    ML = np.log10(peak_wa_mm) + neg_logA0
else:
    ML = np.nan

print(f'Distance correction -log10(A0): {neg_logA0:.2f}')
print(f'Measured ML: {ML:.2f}')
print(f'Catalog magnitude: {ev_mag.magnitude_type} {ev_mag.mag:.1f}')
print(f'\nNote: ML is calibrated for local distances (<600 km). '
      f'At {dist_km:.0f} km this is an extrapolation.')

### Explain

1. Is your $M_L$ estimate close to the catalog magnitude? Why or why not?
2. The Wood-Anderson instrument has a natural period of ~0.8 s (peak response ~1.25 Hz). For a large event, does this band capture the dominant energy? Relate to saturation.

<a id="exercise-b"></a>

---

## Exercise B: $m_b$ — Body-wave magnitude from the P wave

The body-wave magnitude is measured on the **vertical component** in a short-period band (~1 Hz), using the maximum amplitude-to-period ratio of the P wave:

$$
m_b = \log_{10}\!\left(\frac{A}{T}\right) + Q(\Delta, h)
$$

where $A$ is ground displacement amplitude (μm), $T$ is the dominant period (s) of the P phase, and $Q(\Delta, h)$ is an empirical distance-depth attenuation correction (Gutenberg & Richter, 1956; Veith & Clawson, 1972).

### Predict
1. At what magnitude does $m_b$ begin to **saturate**? Why? (Hint: think about corner frequency vs. the 1 Hz measurement band.)
2. Should $m_b$ be measured on the first few cycles of P or anywhere in the P coda?

In [ ]:
# ---- Filter vertical component for mb ----
tr_z = st_disp.copy().select(component='Z')[0]
tr_mb = tr_z.copy()
tr_mb.filter('bandpass', freqmin=0.5, freqmax=2.0, corners=4, zerophase=True)

# Window around P: from predicted P time to P + 30 s (first few cycles)
t_P_abs = ev_time + t_P  # absolute P time
tr_mb_win = tr_mb.copy().trim(t_P_abs - 2, t_P_abs + 30)

times_mb = tr_mb_win.times(reftime=ev_time)
data_mb  = tr_mb_win.data

# Measure max |A| in the P window
idx_peak = np.argmax(np.abs(data_mb))
A_mb_m   = np.abs(data_mb[idx_peak])          # displacement in m
A_mb_um  = A_mb_m * 1e6                       # displacement in μm

# Estimate dominant period: distance between adjacent zero-crossings around peak
sign_changes = np.where(np.diff(np.sign(data_mb)))[0]
if len(sign_changes) >= 2:
    # Find the two zero-crossings bracketing the peak
    zc_before = sign_changes[sign_changes < idx_peak]
    zc_after  = sign_changes[sign_changes > idx_peak]
    if len(zc_before) > 0 and len(zc_after) > 0:
        T_dom = 2.0 * (times_mb[zc_after[0]] - times_mb[zc_before[-1]])
    else:
        T_dom = 1.0  # fallback
else:
    T_dom = 1.0

print(f'P-wave peak displacement: {A_mb_m:.2e} m  =  {A_mb_um:.2f} μm')
print(f'Dominant period: {T_dom:.2f} s')
print(f'A/T = {A_mb_um/T_dom:.2f} μm/s')

In [ ]:
# ---- Plot filtered P waveform with measurement window ----
tr_mb_full = tr_z.copy()
tr_mb_full.filter('bandpass', freqmin=0.5, freqmax=2.0, corners=4, zerophase=True)
times_full = tr_mb_full.times(reftime=ev_time)

fig, ax = plt.subplots()
ax.plot(times_full, tr_mb_full.data * 1e6, 'k', lw=0.5)
ax.axvline(t_P, color='blue', ls='--', label=f'P arrival ({t_P:.0f} s)')
ax.axvspan(t_P - 2, t_P + 30, alpha=0.15, color='orange', label='measurement window')
ax.plot(times_mb[idx_peak], data_mb[idx_peak] * 1e6, 'rv', ms=10, label='peak $A$')
ax.set_xlabel('Time after origin (s)')
ax.set_ylabel('Displacement (μm)')
ax.set_title(f'Vertical BHZ, 0.5–2 Hz — P window for $m_b$')
ax.set_xlim(t_P - 50, t_P + 100)
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# ---- Compute mb ----
# Simplified Q(Δ,h) from Veith & Clawson (1972) — linear approximation
# Q ≈ 0.01Δ + 5.9  for 30° < Δ < 90° and shallow sources (h < 70 km)
# This is a rough approximation; operational agencies use tabulated values.
Q_mb = 0.01 * dist_deg + 5.9

mb = np.log10(A_mb_um / T_dom) + Q_mb

print(f'Q(Δ={dist_deg:.1f}°, h={ev_dep_km:.0f} km) ≈ {Q_mb:.2f}')
print(f'Measured mb: {mb:.2f}')
print(f'Catalog magnitude: {ev_mag.magnitude_type} {ev_mag.mag:.1f}')

### Explain

1. How does your $m_b$ compare to the catalog? Is it higher, lower, or similar?
2. If this event were $M_w$ 8.0 instead of ~6.5, would $m_b$ scale proportionally? Why not? (Hint: corner frequency.)

<a id="exercise-c"></a>

---

## Exercise C: $M_s$ — Surface-wave magnitude from Rayleigh waves

The surface-wave magnitude uses the Rayleigh-wave amplitude at **~20 s period**, measured on the vertical component.  The Prague formula (Vaněk et al., 1962):

$$
M_s = \log_{10}\!\left(\frac{A}{T}\right) + 1.66\,\log_{10}(\Delta) + 3.3
$$

where $A$ is displacement amplitude (μm), $T \approx 20$ s, and $\Delta$ is epicentral distance in degrees.

### Predict
1. Why does $M_s$ require $\Delta > 20°$ (so that higher-mode surface waves have decayed)?
2. For a **deep** event (> 300 km), would you expect $M_s$ to work well? Why?

In [ ]:
# ---- Filter for Rayleigh waves (~20 s) ----
tr_R = tr_z.copy()
tr_R.filter('bandpass', freqmin=0.04, freqmax=0.065, corners=4, zerophase=True)

# Window: Rayleigh group velocity 3.0–4.2 km/s
t_R_abs_early = ev_time + t_R_early
t_R_abs_late  = ev_time + t_R_late
tr_R_win = tr_R.copy().trim(t_R_abs_early, t_R_abs_late)

times_R = tr_R_win.times(reftime=ev_time)
data_R  = tr_R_win.data

# Measure peak amplitude
idx_R_peak = np.argmax(np.abs(data_R))
A_R_m  = np.abs(data_R[idx_R_peak])
A_R_um = A_R_m * 1e6

# Dominant period (same zero-crossing method)
sign_R = np.where(np.diff(np.sign(data_R)))[0]
if len(sign_R) >= 2:
    zc_b = sign_R[sign_R < idx_R_peak]
    zc_a = sign_R[sign_R > idx_R_peak]
    if len(zc_b) > 0 and len(zc_a) > 0:
        T_R = 2.0 * (times_R[zc_a[0]] - times_R[zc_b[-1]])
    else:
        T_R = 20.0  # fallback to nominal
else:
    T_R = 20.0

print(f'Rayleigh peak displacement: {A_R_m:.2e} m  =  {A_R_um:.2f} μm')
print(f'Dominant period: {T_R:.1f} s')
print(f'A/T = {A_R_um/T_R:.2f} μm/s')

In [ ]:
# ---- Plot filtered Rayleigh waveform ----
times_R_full = tr_R.times(reftime=ev_time)

fig, ax = plt.subplots()
ax.plot(times_R_full, tr_R.data * 1e6, 'k', lw=0.5)
ax.axvspan(t_R_early, t_R_late, alpha=0.15, color='green',
           label=f'Rayleigh window ({t_R_early:.0f}–{t_R_late:.0f} s)')
ax.plot(times_R[idx_R_peak], data_R[idx_R_peak] * 1e6, 'rv', ms=10, label='peak $A$')
ax.set_xlabel('Time after origin (s)')
ax.set_ylabel('Displacement (μm)')
ax.set_title(f'Vertical BHZ, 15–25 s period — Rayleigh window for $M_s$')
ax.set_xlim(t_R_early - 100, t_R_late + 200)
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# ---- Compute Ms (Prague formula) ----
Ms = np.log10(A_R_um / T_R) + 1.66 * np.log10(dist_deg) + 3.3

print(f'Distance term: 1.66 * log10({dist_deg:.1f}°) + 3.3 = {1.66*np.log10(dist_deg)+3.3:.2f}')
print(f'Measured Ms: {Ms:.2f}')
print(f'Catalog magnitude: {ev_mag.magnitude_type} {ev_mag.mag:.1f}')

### Explain

1. How does $M_s$ compare to $M_L$ and $m_b$ for this event?
2. $M_s$ measures at ~20 s period. For an event with $f_c \approx 0.05$ Hz (i.e., period ~20 s), is the amplitude still on the low-frequency plateau of the source spectrum?

<a id="exercise-d"></a>

---

## Exercise D: Comparison and wrap-up

We now compare the three magnitude estimates to each other and to the catalog value.

In [ ]:
# ---- Summary table ----
results = pd.DataFrame({
    'Magnitude type': ['$M_L$', '$m_b$', '$M_s$', f'Catalog ({ev_mag.magnitude_type})'],
    'Value': [ML, mb, Ms, ev_mag.mag],
    'Band / method': [
        '1–10 Hz, WA horizontal',
        '0.5–2 Hz, P vertical',
        '15–25 s, Rayleigh vertical',
        'Agency-reported'
    ]
})
results['Value'] = results['Value'].round(2)
display(results)

In [ ]:
# ---- Bar chart comparison ----
fig, ax = plt.subplots(figsize=(6, 4))

labels = ['$M_L$', '$m_b$', '$M_s$']
values = [ML, mb, Ms]
colors = ['#e07b54', '#5485b0', '#6ab06a']

bars = ax.bar(labels, values, color=colors, width=0.5, edgecolor='k')

# Reference line: catalog magnitude
ax.axhline(ev_mag.mag, color='k', ls='--', lw=1.5,
           label=f'Catalog {ev_mag.magnitude_type} = {ev_mag.mag:.1f}')

for bar, val in zip(bars, values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.05,
            f'{val:.2f}', ha='center', va='bottom', fontsize=11)

ax.set_ylabel('Magnitude')
ax.set_title('Measured magnitudes vs catalog')
ax.legend()
ax.set_ylim(min(values) - 1, max(values) + 1)
plt.tight_layout()
plt.show()

## Wrap-up questions

Answer in 2–3 sentences each:

1. **Which magnitude** is closest to the catalog value? Is that expected, given the event size and distance?
2. **$M_L$ at teleseismic distance** — Why is this a poor choice? What would you need to change to make it valid?
3. **$m_b$ vs $M_s$** — If this event had been $M_w$ 8.5, which of the two would you trust more? Why? (Relate to the saturation curves from Lab 7c.)
4. **All three magnitudes agree** (approximately) only in a narrow magnitude range. What range is that, and why?

### Check (hints)
- $M_L$ saturates around $M_w \sim 6.5$ because the WA instrument band (~1 Hz) is above the corner frequency.
- $m_b$ saturates around $M_w \sim 6.5$ for the same reason (short-period measurement).
- $M_s$ saturates around $M_w \sim 8$ because 20 s period eventually falls above the corner frequency of great earthquakes.
- Only $M_w$ (moment magnitude) does not saturate, because it is derived from the low-frequency spectral level ($M_0$).

---

📖 *Shearer (2009), §9.7* — magnitude scales, saturation, and the omega-square model.